In [57]:
from py4j.protocol import NULL_TYPE
from pyspark.sql import SparkSession
from pyspark.sql.functions.builtin import regexp_replace

spark = SparkSession.builder.appName("Read Inside Airbnb Data").getOrCreate()

In [7]:
listings = spark.read.csv(
    "../data/listings.csv.gz",
    sep=",",
    quote='"',
    escape='"',
    header=True,
    inferSchema=True,
    multiLine=True,
    mode="PERMISSIVE"
)

In [11]:
listings.printSchema()

root
 |-- id: long (nullable = true)
 |-- listing_url: string (nullable = true)
 |-- scrape_id: long (nullable = true)
 |-- last_scraped: date (nullable = true)
 |-- source: string (nullable = true)
 |-- name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- neighborhood_overview: string (nullable = true)
 |-- picture_url: string (nullable = true)
 |-- host_id: integer (nullable = true)
 |-- host_url: string (nullable = true)
 |-- host_name: string (nullable = true)
 |-- host_since: date (nullable = true)
 |-- host_location: string (nullable = true)
 |-- host_about: string (nullable = true)
 |-- host_response_time: string (nullable = true)
 |-- host_response_rate: string (nullable = true)
 |-- host_acceptance_rate: string (nullable = true)
 |-- host_is_superhost: string (nullable = true)
 |-- host_thumbnail_url: string (nullable = true)
 |-- host_picture_url: string (nullable = true)
 |-- host_neighbourhood: string (nullable = true)
 |-- host_listings_count: int

In [9]:
# 1. Get a non-null picture URL for any property ("picture_url" field)
# Select any non-null picture URL

listings.select(listings.picture_url).dropna().show(100)

+--------------------+
|         picture_url|
+--------------------+
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0.muscac...|
|https://a0

In [10]:
# 2. Get number of properties that get more than 10 reviews per month

listings.filter(listings.reviews_per_month > 10).select(listings.name).show(100, truncate=False)

+--------------------------------------------------+
|name                                              |
+--------------------------------------------------+
|Locke Studio Apartment at Leman Locke             |
|S - Heathrow Airport Terminal 2 3 4 5 Hatton Cross|
|Feels like home/Family Home                       |
|Micro Studio at Locke at Broken Wharf             |
|City Studio at Locke at Broken Wharf              |
|Private double room with en suite facilities      |
|Bright Studio near Liverpool Street Station       |
|City Studio at Kingsland Locke                    |
|Superior Studio, avg size 23.5 msq                |
|Double Room+ Ensuite                              |
|Heathrow Sleep one / En-Suite rooms ref: #5       |
|Palace retreat  -self contained flat-             |
|D Heathrow Airport Terminals 2 3 4 5 Hatton Cross |
|Cosy Studio at Buckle Street Studios by Locke     |
|Studio Apartment  at Cove Landmark Pinnacle       |
|Covent Garden Studio - Netflix & Nespresso   

In [13]:
# 3. Get a property that has more bathrooms than bedrooms
listings.filter(listings.bathrooms > listings.bedrooms).select('name', 'bathrooms', 'bedrooms')\
    .show(20, truncate=False)


+-------------------------------------------------+---------+--------+
|name                                             |bathrooms|bedrooms|
+-------------------------------------------------+---------+--------+
|Cosy Double studio in Zone 2 Hammersmith (1)     |1.5      |1       |
|LONDON DETACHED HOUSE*ElecGates etc              |2.0      |1       |
|Designer room Park Views 4 mins zone 1 station   |1.5      |1       |
|Maisonette in Central London Zone 1              |1.5      |1       |
|West London,loft ensuite, 5min2tube              |1.5      |1       |
|Shoreditch Loft                                  |1.5      |1       |
|Five minute walk to South Bank                   |1.5      |1       |
|Stunning double room own bathroom                |4.0      |1       |
|Also five minutes to South Bank                  |1.5      |1       |
|Studio 20 min from center                        |1.0      |0       |
|Spacious luxury 2 bedroom apartment              |1.5      |1       |
|Singl

In [55]:
# 4. Get 10 properties where the price is greater than 5,000. Collect the result as a Python list
# Remember to convert a price into a number first!
price_num_df = listings.withColumn('price_num', regexp_replace('price', '[$,]', '')\
                                   .cast('float'))

high_cost_prop = [
    price_num_df.filter(price_num_df.price_num > 5000).select('name', 'price_num').show(10, truncate=False)
    ]
list_of_high_prop = []

for high_cost in high_cost_prop:
    list_of_high_prop.append(high_cost)

print(list_of_high_prop)


+--------------------------------------------------+---------+
|name                                              |price_num|
+--------------------------------------------------+---------+
|Room in a cosy flat. Central, clean               |8000.0   |
|Spacious Private Ground Floor Room                |6309.0   |
|No Longer Available                               |53588.0  |
|Bright & airy DoubleBed with EnSuite in Zone 2!   |74100.0  |
|The Apartments by The Sloane Club, Two Bedroom Apt|7377.0   |
|The Apartments by The Sloane Club, L 2 Bedroom Apt|7377.0   |
|Single room. 7ft x 9ft - Over looking garden      |6523.0   |
|Close To London Eye (TUR)                         |6666.0   |
|Beautiful 2 BR flat in Kilburn with free parking  |6000.0   |
|Semi-detached mews house in Knightsbridge.        |7019.0   |
+--------------------------------------------------+---------+
only showing top 10 rows
[None]


In [42]:
# 5. Get a list of properties with the following characteristics:
# * price < 150
# * more than 20 reviews
# * review_scores_rating > 4.5
# Consider using the "&" operator

price_num_df = listings.withColumn('price_num', regexp_replace('price', '[$,]', '')\
                                   .cast('float'))

price_num_df.filter(
    (price_num_df.price_num < 1500)
    & (price_num_df.number_of_reviews > 20)
    & (price_num_df.review_scores_rating > 4.5))\
    .select('name', 'price_num', 'number_of_reviews', 'review_scores_rating')\
    .show(200, truncate=False)

+--------------------------------------------------+---------+-----------------+--------------------+
|name                                              |price_num|number_of_reviews|review_scores_rating|
+--------------------------------------------------+---------+-----------------+--------------------+
|Holiday London DB Room Let-on going               |70.0     |55               |4.85                |
|Bright Chelsea  Apartment. Chelsea!               |149.0    |97               |4.8                 |
|Very Central Modern 3-Bed/2 Bath By Oxford St W1  |411.0    |56               |4.77                |
|Kew Gardens 3BR house in cul-de-sac               |280.0    |116              |4.8                 |
|You are GUARANTEED to love this                   |90.0     |730              |4.87                |
|SUNNY ROOM PRIVATE BATHROOM PLUS BREAKFAST        |61.0     |387              |4.77                |
|Short Term Home                                   |340.0    |42               |4.

In [50]:
# 6. Get a list of properties with the following characteristics:
# * price < 150 OR more than one bathroom
# Use the "|" operator to implement the OR operator

price_num_df.filter((price_num_df.price_num < 150) | (price_num_df.bathrooms > 1))\
    .select('name', 'price_num', 'bathrooms').show(50, truncate=False)

+--------------------------------------------------+---------+---------+
|name                                              |price_num|bathrooms|
+--------------------------------------------------+---------+---------+
|Holiday London DB Room Let-on going               |70.0     |1.0      |
|Bright Chelsea  Apartment. Chelsea!               |149.0    |1.0      |
|Very Central Modern 3-Bed/2 Bath By Oxford St W1  |411.0    |2.0      |
|Kew Gardens 3BR house in cul-de-sac               |280.0    |1.5      |
|You are GUARANTEED to love this                   |90.0     |0.0      |
|SUNNY ROOM PRIVATE BATHROOM PLUS BREAKFAST        |61.0     |1.0      |
|Short Term Home                                   |340.0    |2.0      |
|SPACIOUS ROOM IN CONTEMPORARY STYLE FLAT          |49.0     |1.0      |
|Room with a view, shared flat,  central  Bankside |96.0     |1.0      |
|You Will Save Money Here                          |71.0     |1.0      |
|Quiet Comfortable Room in Fulham                  

In [53]:
# 7. Get the highest listing price in this dataset
# Consider using the "max" function from "pyspark.sql.functions"

import pyspark.sql.functions as sf

price_num_df.select(sf.max(price_num_df.price_num)).show(20)

+--------------+
|max(price_num)|
+--------------+
|     1085147.0|
+--------------+



In [ ]:
# 8. Get the name and a price of property with the highest number of reviews per month
# Try to use "collect" method to get the price first, and then use it in a "filter" call

#????????/

In [ ]:
# 9. Get the number of hosts in the dataset


In [ ]:
# 10. Get listings with a first review in 2024
# Consider using the "year" function from "pyspark.sql.functions"